## 6.1 Games & Minimax Search

### 1. Multi-agent environments → Games

Key idea:

In normal search problems (like pathfinding), the environment does not fight back.

In multi-agent environments, other agents affect your outcome.

When:

Agents have conflicting goals

One agent’s gain is another agent’s loss

→ we call this adversarial search

→ which is basically games



### 2. Two-player zero-sum games

We simplify first:

Two players: MAX and MIN

MAX wants to maximize the score

MIN wants to minimize the score

They move alternately

Game ends in a terminal state

Terminal states have utilities:

Example in Tic-Tac-Toe:

+10 → MAX wins

0 → Draw

-10 → MIN wins

This is a zero-sum game:
MAX’s gain = MIN’s loss.


### 3. Tic-Tac-Toe as a Search Problem

A game can be modeled like a search problem:

We need:

Initial state

Actions

Transition model

Terminal test

Utility function

Each board configuration is a state.

From each state:

You can generate new states (legal moves).

This naturally forms a game tree.


### 4. Game Tree

Important concept.

Root = initial board

Children = all possible moves

Levels alternate:

MAX level

MIN level

MAX level

MIN level

Leaves = terminal states

Each leaf has a utility value:

-10

0

+10

Even Tic-Tac-Toe’s full tree is large. That’s why we need a systematic way to choose moves.


### 5. The Core Question

How do we make MAX and MIN play optimally?

We compute something called the minimax value of each node.


### 6. What is Minimax?

Two equivalent ways to understand it:

View 1 (pessimistic view)

The minimax value is:

The worst-case outcome that the opponent can force you into.

So MAX assumes:

MIN will always play optimally.

MIN will always try to hurt MAX.

So MAX prepares for the worst.

View 2 (guarantee view)

The minimax value is:

The best outcome you can guarantee for yourself, assuming the opponent is optimal.

This is extremely important:
You don’t assume the opponent makes mistakes.



### 7. How Minimax is Computed

This is the most important slide.

We compute minimax backwards.

Step 1 — Start from terminal nodes

Assign:

+10

0

-10

Step 2 — Move one level up

If it’s a:

MAX node → choose the maximum of children

MIN node → choose the minimum of children

This is called backtracking.


![alt text](image.png)

### 9. Example: First Move in Tic-Tac-Toe

When you evaluate the whole tree:

The center move has the highest minimax value.

So the optimal first move for MAX is:

→ Place X in the center

Not because it looks nice.

But because when you simulate all possible opponent responses,
it guarantees the best possible outcome.


### 10. Intuition Behind It

Minimax is basically:

“I assume my opponent is perfect.

What is the best guaranteed result I can secure?”

It is a worst-case reasoning strategy.

It is very conservative and very rational.


### 11. Important Concept: Optimal Play

If both players use minimax:

Tic-Tac-Toe ends in a draw.

Chess becomes deterministic given infinite computation.

Games become solvable.



12. The Big Limitation

The tree grows exponentially.

If:

branching factor = b

depth = d

Time complexity:

𝑂(b^d)

Even moderate games become impossible to fully search.

That’s why:

Alpha-beta pruning exists

Heuristics exist

Evaluation functions exist

But that comes next.

In [1]:
from __future__ import annotations
from typing import List, Optional, Tuple

# Board representation:
# - 9 cells, indices 0..8
# - "X" = MAX, "O" = MIN, " " = empty
Board = List[str]

WIN_LINES = [
    (0, 1, 2), (3, 4, 5), (6, 7, 8),  # rows
    (0, 3, 6), (1, 4, 7), (2, 5, 8),  # cols
    (0, 4, 8), (2, 4, 6)              # diags
]

def winner(board: Board) -> Optional[str]:
    """Return 'X' if X wins, 'O' if O wins, else None."""
    for a, b, c in WIN_LINES:
        if board[a] != " " and board[a] == board[b] == board[c]:
            return board[a]
    return None

def is_terminal(board: Board) -> bool:
    """Game over if someone won or no empty squares."""
    return winner(board) is not None or all(cell != " " for cell in board)

def utility(board: Board) -> int:
    """
    Terminal utility:
      +10 if X wins
      -10 if O wins
       0  if draw or non-terminal (only call this at terminal in minimax)
    """
    w = winner(board)
    if w == "X":
        return 10
    if w == "O":
        return -10
    return 0

def legal_moves(board: Board) -> List[int]:
    """Indices of empty squares."""
    return [i for i, cell in enumerate(board) if cell == " "]

def apply_move(board: Board, idx: int, player: str) -> Board:
    """Return a new board with player placed at idx."""
    new_board = board.copy()
    new_board[idx] = player
    return new_board

def minimax(board: Board, player: str) -> Tuple[int, Optional[int]]:
    """
    Minimax search (no depth limit) for Tic-Tac-Toe.
    Returns: (best_value, best_move_index)

    player:
      'X' = MAX (maximize value)
      'O' = MIN (minimize value)
    """
    if is_terminal(board):
        return utility(board), None

    moves = legal_moves(board)

    if player == "X":  # MAX
        best_val = -10**9
        best_move = None
        for m in moves:
            child = apply_move(board, m, "X")
            val, _ = minimax(child, "O")
            if val > best_val:
                best_val = val
                best_move = m
        return best_val, best_move

    else:  # player == "O" -> MIN
        best_val = 10**9
        best_move = None
        for m in moves:
            child = apply_move(board, m, "O")
            val, _ = minimax(child, "X")
            if val < best_val:
                best_val = val
                best_move = m
        return best_val, best_move

# --- Optional: Alpha-Beta pruning (same result, faster) ---

def minimax_ab(board: Board, player: str, alpha: int = -10**9, beta: int = 10**9) -> Tuple[int, Optional[int]]:
    if is_terminal(board):
        return utility(board), None

    moves = legal_moves(board)

    if player == "X":  # MAX
        best_val = -10**9
        best_move = None
        for m in moves:
            child = apply_move(board, m, "X")
            val, _ = minimax_ab(child, "O", alpha, beta)
            if val > best_val:
                best_val = val
                best_move = m
            alpha = max(alpha, best_val)
            if alpha >= beta:  # prune
                break
        return best_val, best_move

    else:  # MIN
        best_val = 10**9
        best_move = None
        for m in moves:
            child = apply_move(board, m, "O")
            val, _ = minimax_ab(child, "X", alpha, beta)
            if val < best_val:
                best_val = val
                best_move = m
            beta = min(beta, best_val)
            if alpha >= beta:  # prune
                break
        return best_val, best_move

# --- Tiny helpers to demo quickly ---

def print_board(board: Board) -> None:
    def row(i): return " | ".join(board[i:i+3])
    print(row(0)); print("--+---+--"); print(row(3)); print("--+---+--"); print(row(6))

def play_game():
    board = [" "] * 9
    current_player = "X"  # AI starts

    while not is_terminal(board):
        print_board(board)
        print()

        if current_player == "X":
            print("AI thinking...")
            _, move = minimax_ab(board, "X")
            board = apply_move(board, move, "X")
            print(f"AI plays at position {move}")
        else:
            move = int(input("Your move (0-8): "))
            if move not in legal_moves(board):
                print("Invalid move, try again.")
                continue
            board = apply_move(board, move, "O")

        current_player = "O" if current_player == "X" else "X"

    print()
    print_board(board)

    w = winner(board)
    if w == "X":
        print("AI wins!")
    elif w == "O":
        print("You win!")
    else:
        print("Draw!")

if __name__ == "__main__":
    play_game()

  |   |  
--+---+--
  |   |  
--+---+--
  |   |  

AI thinking...
AI plays at position 0
X |   |  
--+---+--
  |   |  
--+---+--
  |   |  

X |   |  
--+---+--
  | O |  
--+---+--
  |   |  

AI thinking...
AI plays at position 1
X | X |  
--+---+--
  | O |  
--+---+--
  |   |  

X | X | O
--+---+--
  | O |  
--+---+--
  |   |  

AI thinking...
AI plays at position 6
X | X | O
--+---+--
  | O |  
--+---+--
X |   |  

X | X | O
--+---+--
O | O |  
--+---+--
X |   |  

AI thinking...
AI plays at position 5
X | X | O
--+---+--
O | O | X
--+---+--
X |   |  

X | X | O
--+---+--
O | O | X
--+---+--
X | O |  

AI thinking...
AI plays at position 8

X | X | O
--+---+--
O | O | X
--+---+--
X | O | X
Draw!
